In [1]:
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Set
import csv
import time

# An atom is represented as:
# (predicate_name, tuple_of_arguments)
# Example: Young(Bob) = ("Young", ("Bob",))
Atom = Tuple[str, Tuple[str, ...]]


@dataclass
class Rule:
    conditions: List[Atom]
    conclusion: Atom


def is_variable(term: str) -> bool:
    """Variables begin with ?, for example ?x."""
    return term.startswith("?")


def unify_atoms(
    pattern: Atom,
    fact: Atom,
    substitution: Optional[Dict[str, str]] = None
) -> Optional[Dict[str, str]]:
    """
    Match a rule condition with a known fact.
    Example: Young(?x) with Young(Bob) gives {'?x': 'Bob'}.
    Return None if the two atoms do not match.
    """
    if substitution is None:
        substitution = {}

    pred1, args1 = pattern
    pred2, args2 = fact

    if pred1 != pred2 or len(args1) != len(args2):
        return None

    result = dict(substitution)

    for left, right in zip(args1, args2):
        if is_variable(left):
            if left in result and result[left] != right:
                return None
            result[left] = right
        elif left != right:
            return None

    return result


def apply_substitution(
    atom: Atom,
    substitution: Dict[str, str]
) -> Atom:
    """Replace variables in an atom using the substitution."""
    predicate, arguments = atom
    new_arguments = tuple(substitution.get(arg, arg) for arg in arguments)
    return predicate, new_arguments


def find_rule_matches(rule: Rule, facts: Set[Atom]) -> List[Dict[str, str]]:
    """Find substitutions that make every condition of a rule true."""
    matches = [dict()]

    for condition in rule.conditions:
        next_matches = []

        for current in matches:
            grounded_condition = apply_substitution(condition, current)

            for fact in facts:
                new_substitution = unify_atoms(
                    grounded_condition, fact, current
                )
                if new_substitution is not None:
                    next_matches.append(new_substitution)

        matches = next_matches

        if not matches:
            break

    return matches


def forward_chaining(initial_facts: List[Atom], rules: List[Rule]) -> Set[Atom]:
    """
    Start from known facts and repeatedly apply rules.
    Stop when no new fact can be derived.
    """
    facts = set(initial_facts)
    changed = True

    while changed:
        changed = False

        for rule in rules:
            for substitution in find_rule_matches(rule, facts):
                new_fact = apply_substitution(rule.conclusion, substitution)

                if new_fact not in facts:
                    facts.add(new_fact)
                    changed = True

    return facts


def backward_chaining(
    goal: Atom,
    facts: Set[Atom],
    rules: List[Rule],
    depth: int = 0,
    max_depth: int = 30,
    active_goals: Optional[Set[Atom]] = None
) -> bool:
    """Try to prove a goal by finding a rule that concludes it."""
    if goal in facts:
        return True

    if depth >= max_depth:
        return False

    if active_goals is None:
        active_goals = set()

    if goal in active_goals:
        return False

    new_active = set(active_goals)
    new_active.add(goal)

    for rule in rules:
        substitution = unify_atoms(rule.conclusion, goal)

        if substitution is None:
            continue

        all_conditions_true = True

        for condition in rule.conditions:
            sub_goal = apply_substitution(condition, substitution)

            if not backward_chaining(
                sub_goal,
                facts,
                rules,
                depth + 1,
                max_depth,
                new_active
            ):
                all_conditions_true = False
                break

        if all_conditions_true:
            return True

    return False


def parse_supported_sentence(sentence: str):
    """
    A small demonstration parser.

    Supported examples:
      'Bob is young.' -> Young(Bob)
      'If someone is young, then they are smart.'
          -> Young(?x) implies Smart(?x)
      'If someone is smart and kind, then they are successful.'
          -> Smart(?x) AND Kind(?x) implies Successful(?x)

    This is not a complete English/FOL parser.
    """
    text = sentence.strip().lower().rstrip(".")
    text = text.replace(",", " ")

    # Simple fact pattern: "name is property"
    words = text.split()

    if len(words) == 3 and words[1] == "is":
        person = words[0].capitalize()
        prop = words[2].capitalize()
        return "fact", (prop, (person,))

    templates = {
        "if someone is young then they are smart": Rule(
            [("Young", ("?x",))],
            ("Smart", ("?x",))
        ),
        "if someone is smart and kind then they are successful": Rule(
            [
                ("Smart", ("?x",)),
                ("Kind", ("?x",))
            ],
            ("Successful", ("?x",))
        )
    }

    if text in templates:
        return "rule", templates[text]

    raise ValueError(
        "Sentence pattern not supported by this demo parser: " + sentence
    )


def run_demo():
    """Run the Bob example from the assignment."""
    facts = [
        ("Young", ("Bob",)),
        ("Kind", ("Bob",))
    ]

    rules = [
        Rule(
            [("Young", ("?x",))],
            ("Smart", ("?x",))
        ),
        Rule(
            [
                ("Smart", ("?x",)),
                ("Kind", ("?x",))
            ],
            ("Successful", ("?x",))
        )
    ]

    query = ("Successful", ("Bob",))

    start = time.perf_counter()
    derived_facts = forward_chaining(facts, rules)
    forward_answer = query in derived_facts
    elapsed = time.perf_counter() - start

    backward_answer = backward_chaining(query, set(facts), rules)

    print("Forward chaining answer:", forward_answer)
    print("Backward chaining answer:", backward_answer)
    print("Derived facts:")

    for fact in sorted(derived_facts):
        print(" ", fact)

    print("Inference time (seconds):", elapsed)


def evaluate_examples(examples):
    """
    Each example contains:
      facts: list of Atom
      rules: list of Rule
      query: Atom
      label: expected Boolean answer

    Returns accuracy (percentage) and total inference time.
    """
    correct = 0
    total_time = 0.0

    for item in examples:
        start = time.perf_counter()

        derived = forward_chaining(item["facts"], item["rules"])
        prediction = item["query"] in derived

        total_time += time.perf_counter() - start

        if prediction == item["label"]:
            correct += 1

    accuracy = (correct / len(examples) * 100) if examples else 0.0
    return accuracy, total_time


def create_submission_csv(
    ids,
    predictions,
    filename="submission.csv"
):
    """Write IDs and predicted labels to a CSV file."""
    if len(ids) != len(predictions):
        raise ValueError("IDs and predictions must have equal lengths.")

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["id", "label"])

        for example_id, prediction in zip(ids, predictions):
            label = "True" if prediction else "False"
            writer.writerow([example_id, label])


if __name__ == "__main__":
    # 1. Run the main Bob demonstration.
    run_demo()

    # 2. Evaluate the example provided in the assignment.
    demo_examples = [
        {
            "facts": [
                ("Young", ("Bob",)),
                ("Kind", ("Bob",))
            ],
            "rules": [
                Rule(
                    [("Young", ("?x",))],
                    ("Smart", ("?x",))
                ),
                Rule(
                    [
                        ("Smart", ("?x",)),
                        ("Kind", ("?x",))
                    ],
                    ("Successful", ("?x",))
                )
            ],
            "query": ("Successful", ("Bob",)),
            "label": True
        }
    ]

    accuracy, total_time = evaluate_examples(demo_examples)
    print("Accuracy:", accuracy)
    print("Total inference time:", total_time)

    # 3. Example CSV output (uncomment to create submission.csv).
    # create_submission_csv(["example_1"], [True])


Forward chaining answer: True
Backward chaining answer: True
Derived facts:
  ('Kind', ('Bob',))
  ('Smart', ('Bob',))
  ('Successful', ('Bob',))
  ('Young', ('Bob',))
Inference time (seconds): 2.649999805726111e-05
Accuracy: 100.0
Total inference time: 1.4250006643123925e-05


In [2]:
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Set
import csv
import time

# Atom format: (predicate, arguments)
# Example: Young(Bob) = ("Young", ("Bob",))
Atom = Tuple[str, Tuple[str, ...]]


@dataclass
class Rule:
    conditions: List[Atom]
    conclusion: Atom


def is_variable(term: str) -> bool:
    return term.startswith("?")


def unify_atoms(
    pattern: Atom,
    fact: Atom,
    substitution: Optional[Dict[str, str]] = None
) -> Optional[Dict[str, str]]:
    if substitution is None:
        substitution = {}

    pred1, args1 = pattern
    pred2, args2 = fact

    if pred1 != pred2 or len(args1) != len(args2):
        return None

    result = dict(substitution)

    for left, right in zip(args1, args2):
        if is_variable(left):
            if left in result and result[left] != right:
                return None
            result[left] = right
        elif left != right:
            return None

    return result


def apply_substitution(atom: Atom, substitution: Dict[str, str]) -> Atom:
    predicate, arguments = atom
    return predicate, tuple(substitution.get(arg, arg) for arg in arguments)


def find_rule_matches(rule: Rule, facts: Set[Atom]) -> List[Dict[str, str]]:
    matches = [dict()]

    for condition in rule.conditions:
        next_matches = []

        for current in matches:
            grounded_condition = apply_substitution(condition, current)

            for fact in facts:
                new_substitution = unify_atoms(
                    grounded_condition, fact, current
                )
                if new_substitution is not None:
                    next_matches.append(new_substitution)

        matches = next_matches
        if not matches:
            break

    return matches


def forward_chaining(initial_facts: List[Atom], rules: List[Rule]) -> Set[Atom]:
    facts = set(initial_facts)
    changed = True

    while changed:
        changed = False

        for rule in rules:
            for substitution in find_rule_matches(rule, facts):
                new_fact = apply_substitution(rule.conclusion, substitution)

                if new_fact not in facts:
                    facts.add(new_fact)
                    changed = True

    return facts


def backward_chaining(
    goal: Atom,
    facts: Set[Atom],
    rules: List[Rule],
    depth: int = 0,
    max_depth: int = 30,
    active_goals: Optional[Set[Atom]] = None
) -> bool:
    if goal in facts:
        return True

    if depth >= max_depth:
        return False

    if active_goals is None:
        active_goals = set()

    if goal in active_goals:
        return False

    new_active = set(active_goals)
    new_active.add(goal)

    for rule in rules:
        substitution = unify_atoms(rule.conclusion, goal)

        if substitution is None:
            continue

        if all(
            backward_chaining(
                apply_substitution(condition, substitution),
                facts,
                rules,
                depth + 1,
                max_depth,
                new_active
            )
            for condition in rule.conditions
        ):
            return True

    return False


def evaluate_examples(examples):
    """
    Each test example has:
      facts: initial facts
      rules: inference rules
      query: proposition to check
      label: expected True/False answer

    Accuracy = correct predictions / total examples * 100.
    Inference time is the combined time for all examples.
    """
    correct = 0
    total_time = 0.0

    print("\nINDIVIDUAL TEST RESULTS")
    print("-" * 55)

    for index, item in enumerate(examples, start=1):
        start = time.perf_counter()

        derived_facts = forward_chaining(item["facts"], item["rules"])
        prediction = item["query"] in derived_facts

        elapsed = time.perf_counter() - start
        total_time += elapsed

        is_correct = prediction == item["label"]
        if is_correct:
            correct += 1

        print(f"Example {index}:")
        print("  Query              :", item["query"])
        print("  Expected answer    :", item["label"])
        print("  Predicted answer   :", prediction)
        print("  Correct prediction :", is_correct)
        print(f"  Inference time     : {elapsed:.8f} seconds")
        print()

    accuracy = (correct / len(examples) * 100) if examples else 0.0

    print("-" * 55)
    print(f"Correct predictions : {correct}/{len(examples)}")
    print(f"Accuracy            : {accuracy:.2f}%")
    print(f"Total inference time: {total_time:.8f} seconds")

    return accuracy, total_time


def create_submission_csv(ids, predictions, filename="submission.csv"):
    if len(ids) != len(predictions):
        raise ValueError("IDs and predictions must have equal lengths.")

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["id", "label"])

        for example_id, prediction in zip(ids, predictions):
            writer.writerow([example_id, "True" if prediction else "False"])



# Reusable rule sets
student_rules = [
    Rule([("Young", ("?x",))], ("Smart", ("?x",))),
    Rule(
        [("Smart", ("?x",)), ("Kind", ("?x",))],
        ("Successful", ("?x",))
    )
]

bird_rules = [
    Rule([("Bird", ("?x",))], ("CanFly", ("?x",)))
]

mammal_rules = [
    Rule([("Mammal", ("?x",))], ("WarmBlooded", ("?x",)))
]

demo_examples = [
    {
        # Example 1: Successful(Bob) can be derived.
        "facts": [("Young", ("Bob",)), ("Kind", ("Bob",))],
        "rules": student_rules,
        "query": ("Successful", ("Bob",)),
        "label": True
    },
    {
        # Example 2: Alice is young but not known to be kind.
        # Smart(Alice) is derived, but Successful(Alice) is not.
        "facts": [("Young", ("Alice",))],
        "rules": student_rules,
        "query": ("Successful", ("Alice",)),
        "label": False
    },
    {
        # Example 3: Tweety is a bird, so CanFly(Tweety) is derived.
        "facts": [("Bird", ("Tweety",))],
        "rules": bird_rules,
        "query": ("CanFly", ("Tweety",)),
        "label": True
    },
    {
        # Example 4: A penguin is not given the Bird -> CanFly rule here.
        # The query therefore cannot be derived from this knowledge base.
        "facts": [("Penguin", ("Pingu",))],
        "rules": bird_rules,
        "query": ("CanFly", ("Pingu",)),
        "label": False
    },
    {
        # Example 5: Mammal(Rex) implies WarmBlooded(Rex).
        "facts": [("Mammal", ("Rex",))],
        "rules": mammal_rules,
        "query": ("WarmBlooded", ("Rex",)),
        "label": True
    }
]


if __name__ == "__main__":
    # Run the original Bob example separately.
    print("AI REASONING SYSTEM - MAIN DEMO")
    bob_facts = [("Young", ("Bob",)), ("Kind", ("Bob",))]
    bob_query = ("Successful", ("Bob",))

    bob_derived = forward_chaining(bob_facts, student_rules)
    print("Forward chaining answer:", bob_query in bob_derived)
    print(
        "Backward chaining answer:",
        backward_chaining(bob_query, set(bob_facts), student_rules)
    )
    print("Derived facts:")
    for fact in sorted(bob_derived):
        print(" ", fact)

    # Evaluate the system on five examples, including True and False cases.
    accuracy, total_time = evaluate_examples(demo_examples)

    # Optional: create a CSV containing the predictions for the test examples.
    # Uncomment these lines if a submission CSV is required.
    #
    # predictions = [
    #     item["query"] in forward_chaining(item["facts"], item["rules"])
    #     for item in demo_examples
    # ]
    # create_submission_csv(
    #     [f"example_{i}" for i in range(1, len(demo_examples) + 1)],
    #     predictions
    # )


AI REASONING SYSTEM - MAIN DEMO
Forward chaining answer: True
Backward chaining answer: True
Derived facts:
  ('Kind', ('Bob',))
  ('Smart', ('Bob',))
  ('Successful', ('Bob',))
  ('Young', ('Bob',))

INDIVIDUAL TEST RESULTS
-------------------------------------------------------
Example 1:
  Query              : ('Successful', ('Bob',))
  Expected answer    : True
  Predicted answer   : True
  Correct prediction : True
  Inference time     : 0.00001150 seconds

Example 2:
  Query              : ('Successful', ('Alice',))
  Expected answer    : False
  Predicted answer   : False
  Correct prediction : True
  Inference time     : 0.00000908 seconds

Example 3:
  Query              : ('CanFly', ('Tweety',))
  Expected answer    : True
  Predicted answer   : True
  Correct prediction : True
  Inference time     : 0.00000433 seconds

Example 4:
  Query              : ('CanFly', ('Pingu',))
  Expected answer    : False
  Predicted answer   : False
  Correct prediction : True
  Inference tim